<a href="https://colab.research.google.com/github/apeksha300/NPL_Assignment/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gradio spacy nltk pandas matplotlib
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 59.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import gradio as gr
import spacy
import nltk
import pandas as pd
import matplotlib.pyplot as plt

from nltk.stem import PorterStemmer
from spacy import displacy

In [3]:
nlp = spacy.load("en_core_web_sm")

stemmer = PorterStemmer()

print("NLP model loaded successfully!")

NLP model loaded successfully!


In [6]:
def analyze_text(text, operation):

    if not text.strip():
        return "<h3>Please enter some text.</h3>"

    doc = nlp(text)
    if operation == "Tokenization":

        tokens = [token.text for token in doc]

        html = "<h2> Tokenization</h2>"
        html += "<table border='1' style='width:100%; border-collapse:collapse;'>"
        html += "<tr><th>Token</th></tr>"

        for token in tokens:
            html += f"<tr><td>{token}</td></tr>"

        html += "</table>"

        return html

    elif operation == "Stopwords Removal":

        words = []

        for token in doc:
            if not token.is_stop and not token.is_punct:
                words.append(token.text)

        html = "<h2> Stopwords Removal</h2>"

        html += "<p><b>Original Text:</b></p>"
        html += f"<p>{text}</p>"

        html += "<p><b>After Removing Stopwords:</b></p>"
        html += f"<p>{' '.join(words)}</p>"

        return html

    if operation == "Named-Entity Relationship":

        data = []

        for ent in doc.ents:

            data.append({
                "Entity": ent.text,
                "Label": ent.label_,
                "Description": spacy.explain(ent.label_)
            })

        if not data:
            return "<h3>No named entities found.</h3>"

        df = pd.DataFrame(data)

        return df.to_html(
            index=False,
            classes="result-table",
            border=0
        )




    elif operation == "POS Tagging":

        data = []

        for token in doc:

            if not token.is_space:

                data.append({
                    "Word": token.text,
                    "POS": token.pos_,
                    "Detailed Tag": token.tag_,
                    "Lemma": token.lemma_
                })

        df = pd.DataFrame(data)

        return df.to_html(
            index=False,
            classes="result-table",
            border=0
        )




    elif operation == "POS Distribution":

        pos_counts = {}

        for token in doc:

            if not token.is_space:

                pos = token.pos_

                if pos in pos_counts:
                    pos_counts[pos] += 1
                else:
                    pos_counts[pos] = 1

        df = pd.DataFrame(
            list(pos_counts.items()),
            columns=["POS", "Frequency"]
        )


        fig, ax = plt.subplots(figsize=(8, 5))

        ax.bar(
            df["POS"],
            df["Frequency"]
        )

        ax.set_title("POS Distribution")
        ax.set_xlabel("Part of Speech")
        ax.set_ylabel("Frequency")

        plt.xticks(rotation=45)
        plt.tight_layout()


        import io
        import base64

        buffer = io.BytesIO()

        fig.savefig(
            buffer,
            format="png",
            bbox_inches="tight"
        )

        plt.close(fig)

        image_base64 = base64.b64encode(
            buffer.getvalue()
        ).decode()

        table = df.to_html(
            index=False,
            classes="result-table",
            border=0
        )

        return f"""
        <h3>POS Frequency Table</h3>

        {table}

        <h3>POS Distribution Chart</h3>

        <div style="
            background:white;
            padding:20px;
            border-radius:10px;
            text-align:center;
        ">

            <img
                src="data:image/png;base64,{image_base64}"
                style="width:80%;max-width:800px;"
            >

        </div>
        """




    elif operation == "Lemmatization":

        data = []

        for token in doc:

            if not token.is_space:

                data.append({
                    "Original Word": token.text,
                    "Lemma": token.lemma_,
                    "POS": token.pos_
                })

        df = pd.DataFrame(data)

        return df.to_html(
            index=False,
            classes="result-table",
            border=0
        )




    elif operation == "Stemming":

        data = []

        for token in doc:

            if token.is_alpha:

                data.append({
                    "Original Word": token.text,
                    "Stem": stemmer.stem(token.text)
                })

        df = pd.DataFrame(data)

        return df.to_html(
            index=False,
            classes="result-table",
            border=0
        )




    elif operation == "Morphology":

        data = []

        for token in doc:

            if not token.is_space:

                data.append({
                    "Word": token.text,
                    "Lemma": token.lemma_,
                    "POS": token.pos_,
                    "Morphology": str(token.morph)
                })

        df = pd.DataFrame(data)

        return df.to_html(
            index=False,
            classes="result-table",
            border=0
        )




    elif operation == "Dependencies":


        svg = displacy.render(
            doc,
            style="dep",
            jupyter=False,
            options={
                "distance": 120,
                "compact": False
            }
        )


        svg = svg.replace(
            "</svg>",
            """
            <style>
                text {
                    fill: #111111 !important;
                    font-family: Arial, sans-serif !important;
                }

                path {
                    stroke: #222222 !important;
                }

                marker path {
                    fill: #222222 !important;
                    stroke: #222222 !important;
                }
            </style>
            </svg>
            """
        )


        import base64

        svg_base64 = base64.b64encode(
            svg.encode("utf-8")
        ).decode("utf-8")


        data = []

        for token in doc:

            if not token.is_space:

                data.append({
                    "Word": token.text,
                    "POS": token.pos_,
                    "Dependency": token.dep_,
                    "Head": token.head.text
                })

        df = pd.DataFrame(data)

        table = df.to_html(
            index=False,
            classes="result-table",
            border=0
        )

        return f"""
        <h3>Dependency Tree</h3>

        <div style="
            background:white;
            padding:20px;
            border-radius:10px;
            overflow-x:auto;
            text-align:center;
        ">

            <img
                src="data:image/svg+xml;base64,{svg_base64}"
                style="
                    width:100%;
                    max-width:1100px;
                    height:auto;
                "
            >

        </div>

        <h3>Dependency Information</h3>

        {table}
        """

In [7]:
css = """
body {
    background-color: #f5f7fb;
}

h1 {
    text-align: center;
    color: #1f2937;
}

.subtitle {
    text-align: center;
    color: #6b7280;
    font-size: 17px;
}

.result-table {
    width: 100%;
    border-collapse: collapse;
    margin-top: 10px;
}

.result-table th {
    background-color: #4f46e5;
    color: white;
    padding: 10px;
    text-align: left;
}

.result-table td {
    padding: 9px;
    border-bottom: 1px solid #ddd;
}
"""

In [8]:
with gr.Blocks(
    title="NLP Analysis Platform",
    css=css
) as demo:



    gr.Markdown("""
<div class="title"> NLP Explorer</div>
<div class="subtitle">
Interactive Natural Language Processing Dashboard
</div>
""")




    gr.Markdown("##  Enter Text")

    text_input = gr.Textbox(
        label="Input Text",
        placeholder="Enter an English sentence or paragraph...",
        lines=6
    )




    operation = gr.Dropdown(
    choices=[
        "Tokenization",
        "Stopwords Removal",
        "POS Tagging",
        "Lemmatization",
        "Named Entity Recognition",
        "Stemming",
        "Relationship / Dependencies"
    ],
    value="Tokenization",
    label="Select NLP Operation"
)



    analyze_button = gr.Button(
        " Analyze Text",
        variant="primary"
    )



    gr.Markdown("##  Result")

    output = gr.HTML()




    analyze_button.click(
        fn=analyze_text,
        inputs=[text_input, operation],
        outputs=output
    )




    gr.Markdown(
        """
        ---

        **NLP Analysis Platform**

        Python • spaCy • NLTK • Gradio
        """
    )

/tmp/ipykernel_1360/2177237855.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


In [9]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6e2a95efe67932936d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
